# Session 13: Curve fitting

A laboratory validates its line-broadening method on a certified standard before it reports nanoparticle sizes. It fits a profile to one diffraction line of NIST SRM 1979 zinc oxide, turns the fitted integral breadth into a crystallite size by the Scherrer equation, and asks whether that size falls inside the certified interval for the reflection. This notebook fits profiles with `curve_fit`, from a starting guess and bounds taken from the physics, and reads the residuals before it trusts a size. Section 1 works the method on Dataset A, one reflection of one pattern. Section 2 applies it to your own reflection, with the mixed profile written by you. Section 3 compares two profiles on your reflection and asks whether your call depends on the choice. Section 4 is the judgment cell.

The data are real: the five synchrotron powder diffraction patterns of the 15 nm powder that NIST used to certify SRM 1979, from NIST, [doi:10.18434/M32269](https://doi.org/10.18434/M32269), public, accessed 2026-10-23. `data/README.md` describes the files, the certificate table and who has which reflection.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 13 assignment on Canvas by 1:30 pm on Tuesday, November 17. Submissions are individual. Every student's pattern is in `data/`, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart and holds your prompt record from section 2, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `warnings` and `Path` from the standard library, NumPy, pandas and Matplotlib, and one new function. `curve_fit` from `scipy.optimize` finds the parameters of a model function that best fit measured points, and returns them with their covariance matrix. `warnings` lets one cell print a warning SciPy raises as a line of output. The cell prints nothing when it works.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Parameters

The data files; the wavelength and the certified lattice of the 15 nm powder, which fix where every reflection sits; the nominal size on the label, which fixes how wide a line to expect; how far a fit may move a centroid; the starting mixing fraction; the coverage of the call; then Dataset A, and your pattern and reflection, set in section 2. Leave `PATTERN` and `REFLECTION` as `CHANGE-ME` until section 2.

In [ ]:
DATA_DIR = Path("data")                        # the five patterns, the certificate, the window edges and the assignment
CERTIFICATE = DATA_DIR / "certificate.csv"     # certificate Table 1: hkl, position, <L>vol (nm) and U (nm, k = 2), the stacking-fault mark
WINDOWS = DATA_DIR / "windows.csv"             # each reflection's window on each pattern, degrees 2theta, halfway to the neighbouring lines
REFLECTIONS = DATA_DIR / "reflections.csv"     # every student's pattern and reflection, by student number, section 2
WAVELENGTH_A = 0.41419        # A, the X-ray wavelength refined in the NIST files (APS beamline 11-BM)
LATTICE_A_A = 3.249664        # A, the certified a of the 15 nm powder (certificate Table 3)
LATTICE_C_A = 5.208330        # A, the certified c of the 15 nm powder
NOMINAL_NM = 15.0             # nm, the size on the SRM's label: the only size known before a fit, used for every starting guess
POSITION_TOL_DEG = 0.05       # deg 2theta, how far a fitted centroid may move from the lattice position
FWHM_MAX_DEG = 1.0            # deg 2theta, the widest profile a fit may return
EDGE_POINTS = 10              # points at each end of a window whose median is the background's starting guess
ETA_START = 0.5               # the pseudo-Voigt's starting mixing fraction: halfway between Gaussian and Lorentzian
COVERAGE = 2                  # the call uses L +- COVERAGE standard errors against <L>vol +- U
EDGE_FRACTION = 0.1           # the share of a window at each end over which section 3 averages the residuals, where a neighbour's tail shows
PATTERN_A = "5857"            # Dataset A, section 1: pattern 5857, reflection 032
REFLECTION_A = "032"
PATTERN = "CHANGE-ME"          # your pattern, from data/reflections.csv by your student number, section 2
REFLECTION = "CHANGE-ME"    # your reflection, from the same row, section 2

## 1. Explore: a profile fitted to one reflection, and the size it gives

Dataset A is reflection 032 of pattern 5857, one of the five patterns NIST measured to certify the standard. The question for this section is what a fitted profile's integral breadth says about the crystallite size, and how the residuals tell a wrong profile from a right one. The first cell defines `read_cif`, which returns a pattern's 2θ axis, counts and σ, `position_deg`, the position of any reflection from the certified lattice, and `breadth_deg`, the breadth a given crystallite size produces. It cuts the window `windows.csv` gives for 032, prints what it holds, and plots the pattern whole and the window.

In [ ]:
def read_cif(path):
    """A NIST SRM 1979 pattern, its angle-corrected block: 2theta (deg), counts, and sigma = 1 / sqrt(weight) (counts)."""
    lines = Path(path).read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip() == "_pd_proc_2theta_corrected")
    rows = []
    for line in lines[start + 3:]:          # the loop's three column names, then one point per line until the next block
        if not line.strip() or line.lstrip().startswith(("_", "loop_", "data_")):
            break
        two_theta, counts, weight = (float(v) for v in line.split()[:3])
        rows.append((two_theta, counts, weight))
    table = np.array(rows)
    return table[:, 0], table[:, 1], 1 / np.sqrt(table[:, 2])


def cif_path(pattern):
    """The file of a pattern, named by its NIST scan number, for example 5857 -> data/5857_15_nm_174.cif."""
    found = sorted(DATA_DIR.glob(f"{pattern}_15_nm_*.cif"))
    if not found:
        raise FileNotFoundError(f"no {DATA_DIR}/{pattern}_15_nm_*.cif; set PATTERN in Parameters from data/reflections.csv")
    return found[0]


def position_deg(hkl):
    """2theta (deg) of a hexagonal reflection from the certified lattice: 1/d^2 = 4/3 (h^2 + hk + k^2)/a^2 + l^2/c^2."""
    h, k, l = (int(c) for c in hkl)
    inv_d2 = 4 / 3 * (h * h + h * k + k * k) / LATTICE_A_A ** 2 + l * l / LATTICE_C_A ** 2
    return 2 * np.degrees(np.arcsin(WAVELENGTH_A * np.sqrt(inv_d2) / 2))


def breadth_deg(size_nm, two_theta):
    """Scherrer read backwards: the integral breadth (deg 2theta) a crystallite size (nm) gives at two_theta."""
    return np.degrees(WAVELENGTH_A / (size_nm * 10 * np.cos(np.radians(two_theta / 2))))


def window(pattern, hkl):
    """One reflection's window of a pattern, by windows.csv: 2theta (deg), counts, sigma (counts), and the edges."""
    two_theta, counts, sigma = read_cif(cif_path(pattern))
    edges = pd.read_csv(WINDOWS, dtype={"pattern": str, "hkl": str}).set_index(["pattern", "hkl"])
    lo, hi = edges.loc[(pattern, hkl), ["lo_deg", "hi_deg"]]
    inside = (two_theta > lo) & (two_theta < hi)
    return two_theta[inside], counts[inside], sigma[inside], (lo, hi)


certificate = pd.read_csv(CERTIFICATE, dtype={"hkl": str}).set_index("hkl")
tt_all, counts_all, sigma_all = read_cif(cif_path(PATTERN_A))
tt_a, counts_a, sigma_a, (lo_a, hi_a) = window(PATTERN_A, REFLECTION_A)
pos_a = position_deg(REFLECTION_A)
cert_a, u_a = certificate.loc[REFLECTION_A, "L_vol_nm"], certificate.loc[REFLECTION_A, "U_nm"]
print(f"pattern {PATTERN_A}: {tt_all.size} points from {tt_all[0]:.3f} to {tt_all[-1]:.3f} deg 2theta, {tt_all[1] - tt_all[0]:.3f} deg apart; "
      f"strongest {counts_all.max():.0f} counts at {tt_all[np.argmax(counts_all)]:.3f} deg")
print(f"the {REFLECTION_A} window, {lo_a:.3f} to {hi_a:.3f} deg: {tt_a.size} points; peak {counts_a.max():.0f} counts at {tt_a[np.argmax(counts_a)]:.4f} deg "
      f"over a background near {np.median(np.r_[counts_a[:EDGE_POINTS], counts_a[-EDGE_POINTS:]]):.0f}; sigma {sigma_a.min():.1f} to {sigma_a.max():.1f} counts")
print(f"lattice position of {REFLECTION_A}: {pos_a:.4f} deg; a {NOMINAL_NM:.0f} nm crystallite gives an integral breadth of {breadth_deg(NOMINAL_NM, pos_a):.3f} deg there")
print(f"certificate, {REFLECTION_A}: <L>vol {cert_a} +- {u_a} nm (k = 2); unaffected by stacking faults: {certificate.loc[REFLECTION_A, 'unaffected_by_stacking_faults']}")

fig, (ax_all, ax_win) = plt.subplots(1, 2, figsize=(10, 4))
ax_all.plot(tt_all, counts_all, color="black", linewidth=0.6)
ax_all.axvspan(lo_a, hi_a, color="gray", alpha=0.3, label=f"the {REFLECTION_A} window")
ax_all.set_xlabel("2θ (deg)")
ax_all.set_ylabel("counts")
ax_all.set_title(f"pattern {PATTERN_A}, the 15 nm powder")
ax_all.legend()
ax_win.plot(tt_a, counts_a, "k.", markersize=2, label="measured")
ax_win.axvline(pos_a, color="gray", linestyle=":", label="lattice position")
ax_win.set_xlabel("2θ (deg)")
ax_win.set_ylabel("counts")
ax_win.set_title(f"Dataset A: reflection {REFLECTION_A} of pattern {PATTERN_A}")
ax_win.legend()
fig.tight_layout()
plt.show()

Pattern 5857 runs from 2.0 to 47.5° 2θ, its strongest line 22,432 counts at 9.59° (the 011 reflection), with the lines weakening and crowding toward higher angle; the 032 window, 26.77 to 27.37°, holds one peak of 1,334 counts at 27.13° over a background near 130, with tails reaching about 0.15° either side of the dotted lattice position at 27.14°.

A **Gaussian** on a linear background has five parameters: area, centroid, full width at half maximum (FWHM), and the background's level and slope. Its **integral breadth** β is the area over the peak height, and the Scherrer equation turns β, in radians, into a size, L = λ / (β cos θ), whose standard error follows from the covariance matrix. `curve_fit` needs a **starting guess**, `p0`, and without one starts every parameter at 1. The next cell defines the profile, breadth and size, fits once with no guess, prints SciPy's warning, then fits from a guess built from the physics (the lattice position, a 15 nm crystallite's breadth, height times breadth, the window's ends) with **bounds** on the centroid.

In [ ]:
def gaussian(x, area, x0, fwhm, b0, b1):
    """A Gaussian of unit area times area and FWHM fwhm, centred at x0, on a linear background b0 + b1 (x - x0)."""
    s = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return area / (s * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((x - x0) / s) ** 2) + b0 + b1 * (x - x0)


def lorentzian(x, area, x0, fwhm, b0, b1):
    """A Lorentzian of unit area times area and FWHM fwhm, centred at x0, on the same linear background."""
    return area * (fwhm / (2 * np.pi)) / ((x - x0) ** 2 + (fwhm / 2) ** 2) + b0 + b1 * (x - x0)


def integral_breadth_deg(model, popt):
    """The fitted profile's area over its height above background, deg 2theta: popt is (area, x0, fwhm, ..., b0, b1)."""
    height = model(popt[1], *popt[:-2], 0, 0)
    return popt[0] / height


def size_nm(model, popt, pcov):
    """L = lambda / (beta cos theta) in nm, and its standard error by linear propagation of pcov through L."""
    def size(p):
        return WAVELENGTH_A / (np.radians(integral_breadth_deg(model, p)) * np.cos(np.radians(p[1] / 2))) / 10
    L = size(popt)
    steps = 1e-6 * np.maximum(np.abs(popt), 1e-3)
    gradient = np.array([(size(popt + steps[j] * np.eye(len(popt))[j]) - L) / steps[j] for j in range(len(popt))])
    return L, float(np.sqrt(gradient @ pcov @ gradient))


def call(L, se, L_cert, U):
    """pass when L +- COVERAGE se lies inside L_cert +- U, fail when it lies outside, else undecided."""
    low, high = L - COVERAGE * se, L + COVERAGE * se
    if low > L_cert - U and high < L_cert + U:
        return "pass"
    if high < L_cert - U or low > L_cert + U:
        return "fail"
    return "undecided"


def guess_and_bounds(two_theta, counts, position):
    """A starting guess and bounds for a profile on a window, from the lattice position and the nominal size."""
    background = np.median(np.r_[counts[:EDGE_POINTS], counts[-EDGE_POINTS:]])
    breadth = breadth_deg(NOMINAL_NM, position)
    p0 = [(counts.max() - background) * breadth, position, breadth, background, 0.0]
    lower = [0, position - POSITION_TOL_DEG, 1e-4, -np.inf, -np.inf]
    upper = [np.inf, position + POSITION_TOL_DEG, FWHM_MAX_DEG, np.inf, np.inf]
    return p0, (lower, upper)


with warnings.catch_warnings(record=True) as caught:        # SciPy warns about this fit; the warning is printed as a line, not raised
    warnings.simplefilter("always")
    try:
        popt_none, _ = curve_fit(gaussian, tt_a, counts_a, sigma=sigma_a, absolute_sigma=True)
        print(f"Gaussian with no starting guess: centroid {popt_none[1]:.3f} deg, FWHM {popt_none[2]:.4f} deg, area {popt_none[0]:.0f}: "
              f"the optimizer stopped where it could, {abs(popt_none[1] - pos_a):.2f} deg from the lattice position")
    except RuntimeError as err:
        print(f"Gaussian with no starting guess: {err}")
for w in caught:
    print(f"SciPy's warning for that fit: {w.message}")

p0_a, bounds_a = guess_and_bounds(tt_a, counts_a, pos_a)
print(f"the guess from the physics: area {p0_a[0]:.0f}, centroid {p0_a[1]:.4f} deg, FWHM {p0_a[2]:.4f} deg, background {p0_a[3]:.0f}, slope {p0_a[4]}")
popt_g, pcov_g = curve_fit(gaussian, tt_a, counts_a, p0=p0_a, sigma=sigma_a, absolute_sigma=True, bounds=bounds_a)
L_g, se_g = size_nm(gaussian, popt_g, pcov_g)
resid_g = (counts_a - gaussian(tt_a, *popt_g)) / sigma_a
chi2_g = np.sum(resid_g ** 2) / (tt_a.size - len(popt_g))
print(f"Gaussian: centroid {popt_g[1]:.4f} deg ({popt_g[1] - pos_a:+.4f} from the lattice), FWHM {popt_g[2]:.4f} deg, integral breadth {integral_breadth_deg(gaussian, popt_g):.4f} deg")
print(f"L = {L_g:.2f} +- {se_g:.2f} nm against the certificate's {cert_a} +- {u_a} nm: {call(L_g, se_g, cert_a, u_a)}; reduced chi-squared {chi2_g:.2f}")

The fit's standard error is a fraction of a percent, and the curve overlays the points, yet the size misses the certificate. The **residuals**, measured minus fitted and divided by each point's σ, show where the profile fails. A Gaussian falls to zero too fast, so the measured tails sit above it and the shoulders below. A **Lorentzian** has the heavy tails that size broadening produces. The next cell fits it from the same guess, prints both profiles' sizes against the certificate, and draws the two fits with their residuals.

In [ ]:
popt_l, pcov_l = curve_fit(lorentzian, tt_a, counts_a, p0=p0_a, sigma=sigma_a, absolute_sigma=True, bounds=bounds_a)
L_l, se_l = size_nm(lorentzian, popt_l, pcov_l)
resid_l = (counts_a - lorentzian(tt_a, *popt_l)) / sigma_a
chi2_l = np.sum(resid_l ** 2) / (tt_a.size - len(popt_l))
profiles_a = pd.DataFrame({"L_nm": [L_g, L_l], "se_nm": [se_g, se_l], "call": [call(L_g, se_g, cert_a, u_a), call(L_l, se_l, cert_a, u_a)],
                           "breadth_deg": [integral_breadth_deg(gaussian, popt_g), integral_breadth_deg(lorentzian, popt_l)],
                           "reduced_chi2": [chi2_g, chi2_l]}, index=["Gaussian", "Lorentzian"])
print(f"certificate, {REFLECTION_A}: {cert_a} +- {u_a} nm")
print(profiles_a.round({"L_nm": 2, "se_nm": 2, "breadth_deg": 4, "reduced_chi2": 2}).to_string())

fig, (ax_fit, ax_res) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax_fit.plot(tt_a, counts_a, "k.", markersize=2, label="measured")
ax_fit.plot(tt_a, gaussian(tt_a, *popt_g), linestyle="--", label=f"Gaussian, L {L_g:.1f} nm")
ax_fit.plot(tt_a, lorentzian(tt_a, *popt_l), linestyle="-", label=f"Lorentzian, L {L_l:.1f} nm")
ax_fit.set_ylabel("counts")
ax_fit.set_title(f"Dataset A: reflection {REFLECTION_A} of pattern {PATTERN_A}, two profiles")
ax_fit.legend()
ax_res.plot(tt_a, resid_g, linestyle="--", label="Gaussian")
ax_res.plot(tt_a, resid_l, linestyle="-", label="Lorentzian")
ax_res.axhline(0, color="gray")
ax_res.set_xlabel("2θ (deg)")
ax_res.set_ylabel("residual / σ")
ax_res.legend()
fig.tight_layout()
plt.show()

Both curves overlay the peak at a glance, but the Gaussian (dashed) tops out near 1,180 counts and meets the background by 26.95 and 27.30°, whereas the Lorentzian (solid) follows the 1,334-count top and the tails; the Gaussian's residuals swing in bands between about +4σ at the top and on both flanks and −4σ on the shoulders between them, and the Lorentzian's stay within about ±2σ with no band.

**Explain.** In two sentences: which feature of the residual plot says that the Gaussian profile, not the sample, is the reason its size misses the certificate, and what would a laboratory that fitted only Gaussians have concluded about its method from Dataset A?

*Your answer:*

## 2. Apply: your reflection

Your reflection is one of the 21 the certificate lists apart from Dataset A's, on one of the five patterns: a different line at a different angle, affected or not by stacking faults, with or without a neighbouring line close enough to reach into its window. Section 1's functions carry over unchanged. What changes is the profile: real size-broadened lines sit between the Gaussian and the Lorentzian, and you write the mixture.

**[TASK 1]** Find your student number in `data/reflections.csv` and set `PATTERN` and `REFLECTION` in the Parameters cell to the values beside it, for example `"5858"` and `"014"`. Run the Parameters cell, then the next cell. It reads your pattern, cuts your window, and prints what it holds, the lattice position of your reflection and the certificate's interval. Until `PATTERN` is set, it stops with `FileNotFoundError: no data/CHANGE-ME_15_nm_*.cif`.

In [ ]:
tt, counts, sigma, (lo, hi) = window(PATTERN, REFLECTION)
pos = position_deg(REFLECTION)
L_cert, U_cert = certificate.loc[REFLECTION, "L_vol_nm"], certificate.loc[REFLECTION, "U_nm"]
print(f"reflection {REFLECTION} of pattern {PATTERN}, window {lo:.3f} to {hi:.3f} deg: {tt.size} points; peak {counts.max():.0f} counts at {tt[np.argmax(counts)]:.4f} deg "
      f"over a background near {np.median(np.r_[counts[:EDGE_POINTS], counts[-EDGE_POINTS:]]):.0f}")
print(f"lattice position {pos:.4f} deg; a {NOMINAL_NM:.0f} nm crystallite gives an integral breadth of {breadth_deg(NOMINAL_NM, pos):.3f} deg there")
print(f"certificate, {REFLECTION}: <L>vol {L_cert} +- {U_cert} nm (k = 2); unaffected by stacking faults: {certificate.loc[REFLECTION, 'unaffected_by_stacking_faults']}")

fig, ax = plt.subplots()
ax.plot(tt, counts, "k.", markersize=2, label="measured")
ax.axvline(pos, color="gray", linestyle=":", label="lattice position")
ax.set_xlabel("2θ (deg)")
ax.set_ylabel("counts")
ax.set_title(f"reflection {REFLECTION} of pattern {PATTERN}")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

A **pseudo-Voigt** is η parts Lorentzian and 1 − η parts Gaussian of the same area, centroid and FWHM, on one linear background, so it has six parameters: area, centroid, FWHM, η, background level and slope, in that order. The next cell builds its starting guess and bounds for your window by section 1's rule, with η starting at 0.5 and bounded between 0 and 1, and prints them. Your cell in TASK 2 uses these names.

In [ ]:
p0_profile, (lower_profile, upper_profile) = guess_and_bounds(tt, counts, pos)
p0_pv = p0_profile[:3] + [ETA_START] + p0_profile[3:]
bounds_pv = (lower_profile[:3] + [0.0] + lower_profile[3:], upper_profile[:3] + [1.0] + upper_profile[3:])
print(f"p0_pv: area {p0_pv[0]:.0f}, centroid {p0_pv[1]:.4f} deg, FWHM {p0_pv[2]:.4f} deg, eta {p0_pv[3]}, background {p0_pv[4]:.0f}, slope {p0_pv[5]}")
print(f"bounds_pv: centroid {bounds_pv[0][1]:.4f} to {bounds_pv[1][1]:.4f} deg, FWHM up to {bounds_pv[1][2]} deg, eta {bounds_pv[0][3]} to {bounds_pv[1][3]}")

**Specific prompts**, today's AI practice, met in session 5. Before TASK 2, ask the assistant in Claude Code, in your own words, for the cell TASK 2 describes. Say which notebook and task, what the function is made of and the order of its parameters, which names the cell must set, how the fit is weighted and bounded, and what you want reported and checked. Send it, and only then read the course's prompt below as an example of what a specific prompt names:

`In activity.ipynb, section 2, TASK 2: write pseudo_voigt(x, area, x0, fwhm, eta, b0, b1) as eta times section 1's lorentzian plus (1 - eta) times gaussian, both of the same area, x0 and fwhm with no background, plus b0 + b1 (x - x0). Fit it to tt and counts with curve_fit from p0_pv with bounds=bounds_pv, sigma=sigma and absolute_sigma=True, and set popt_pv and pcov_pv for the given cell after it. Explain first what the weights and the bounds change, then give the cell. Then tell me the integral breadth in degrees 2theta and L = lambda / (beta cos theta) in nm with its standard error from the covariance matrix, and how I can check them against the given cell.`

Read the answer against the given cell after TASK 2, which recomputes the breadth and the size from your fitted parameters and prints η. An answer that fits well may still add cautions; check each against the certificate table before you accept it. Add a markdown cell under this one (hover just below this cell and click **+ Markdown**) holding these lines, and fill the first now:

```
**Prompt record**

- **My prompt:**
- **What my prompt named, and what the answer added or cautioned beyond it:**
- **What I verified against the given cell, or rejected, and from what:**
```

Fill the second and third lines after TASK 2. Your prompt record is one candidate for part (b) of the judgment cell, and the execution point needs it.

**[TASK 2]** In a code cell you add under this one (**+ Code**), define `pseudo_voigt(x, area, x0, fwhm, eta, b0, b1)` from section 1's `gaussian` and `lorentzian`, η parts Lorentzian and 1 − η parts Gaussian of the same area, centroid and FWHM, with the background added once, and fit it to your window with `curve_fit`, from `p0_pv` with `bounds=bounds_pv`, weighted by `sigma` with `absolute_sigma=True`. The cell sets two names, `popt_pv` and `pcov_pv`. The cell after it prints η, the FWHM, the integral breadth, the size with its standard error, the call and the reduced chi-squared, and stops with `NameError: name 'pseudo_voigt' is not defined` until your cell has run.

In [ ]:
L_pv, se_pv = size_nm(pseudo_voigt, popt_pv, pcov_pv)
resid_pv = (counts - pseudo_voigt(tt, *popt_pv)) / sigma
chi2_pv = np.sum(resid_pv ** 2) / (tt.size - len(popt_pv))
print(f"reflection {REFLECTION} of pattern {PATTERN}, pseudo-Voigt: eta {popt_pv[3]:.2f}, centroid {popt_pv[1]:.4f} deg ({popt_pv[1] - pos:+.4f} from the lattice), "
      f"FWHM {popt_pv[2]:.4f} deg, integral breadth {integral_breadth_deg(pseudo_voigt, popt_pv):.4f} deg")
print(f"L = {L_pv:.2f} +- {se_pv:.2f} nm against the certificate's {L_cert} +- {U_cert} nm: {call(L_pv, se_pv, L_cert, U_cert)}; "
      f"margin to the nearer edge {U_cert - abs(L_pv - L_cert):.2f} nm; reduced chi-squared {chi2_pv:.2f}")
if popt_pv[3] < 0.005 or popt_pv[3] > 0.995:
    print("eta sits at a bound: the data want a profile outside the family, and the standard error is optimistic there")

**Explain.** Before you compare: in two sentences, give your call, your L ± 2 se against the certified interval with the margin to its nearer edge in nm, and your η. Then compare with a neighbor: set both reflections' η side by side, and say whose is higher and whether the certificate marks that reflection as affected by stacking faults. Add one sentence on where your two answers differ and what in your data explains it.

*Your answer:*

## 3. Analyze: which profile the residuals justify

A Gaussian is the profile most fitting software offers first, and on a certified standard it is a defensible choice until the residuals say otherwise. The question for this section is whether your call depends on the profile, and what a laboratory that fitted Gaussians would have certified on your reflection. The next cell fits the Gaussian to your window from the same guess without η, prints both profiles' sizes and calls, says whether a certified neighbouring line reaches into your window and how the pseudo-Voigt's residuals sit at the window's two edges, where a neighbour's tail shows, and draws the two residual plots one above the other.

In [ ]:
popt_g2, pcov_g2 = curve_fit(gaussian, tt, counts, p0=p0_profile, sigma=sigma, absolute_sigma=True, bounds=(lower_profile, upper_profile))
L_g2, se_g2 = size_nm(gaussian, popt_g2, pcov_g2)
resid_g2 = (counts - gaussian(tt, *popt_g2)) / sigma
chi2_g2 = np.sum(resid_g2 ** 2) / (tt.size - len(popt_g2))
profiles = pd.DataFrame({"L_nm": [L_g2, L_pv], "se_nm": [se_g2, se_pv], "call": [call(L_g2, se_g2, L_cert, U_cert), call(L_pv, se_pv, L_cert, U_cert)],
                         "breadth_deg": [integral_breadth_deg(gaussian, popt_g2), integral_breadth_deg(pseudo_voigt, popt_pv)],
                         "reduced_chi2": [chi2_g2, chi2_pv]}, index=["Gaussian", "pseudo-Voigt"])
print(f"reflection {REFLECTION} of pattern {PATTERN}; certificate {L_cert} +- {U_cert} nm")
print(profiles.round({"L_nm": 2, "se_nm": 2, "breadth_deg": 4, "reduced_chi2": 2}).to_string())
print(f"the call {'changes' if profiles.call.iloc[0] != profiles.call.iloc[1] else 'does not change'} with the profile; "
      f"the Gaussian reads L {(L_g2 / L_pv - 1) * 100:+.1f}% against the pseudo-Voigt")

others = certificate.drop(index=REFLECTION)
reach = breadth_deg(NOMINAL_NM, others["two_theta_deg"]) * 2                      # a line reaches about two breadths from its centre
inside = others[(others["two_theta_deg"] + reach > lo) & (others["two_theta_deg"] - reach < hi)]
if len(inside):
    nearest = inside.iloc[np.argmin(np.abs(inside["two_theta_deg"] - pos))]
    print(f"a certified neighbour reaches into this window: {nearest.name} at {nearest['two_theta_deg']:.3f} deg, {abs(nearest['two_theta_deg'] - pos):.3f} deg away")
else:
    print("no certified line reaches into this window")
edge = max(int(EDGE_FRACTION * tt.size), 1)
print(f"the pseudo-Voigt residuals' mean over the outer {EDGE_FRACTION:.0%} of the window: left edge {resid_pv[:edge].mean():+.1f} sigma, "
      f"right edge {resid_pv[-edge:].mean():+.1f} sigma (a flat fit leaves both near 0; a neighbour's tail lifts one or both)")

fig, (ax_g, ax_pv) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
ax_g.plot(tt, resid_g2, linestyle="--", label=f"Gaussian, reduced chi-squared {chi2_g2:.1f}")
ax_pv.plot(tt, resid_pv, linestyle="-", label=f"pseudo-Voigt, reduced chi-squared {chi2_pv:.1f}")
for ax in (ax_g, ax_pv):
    ax.axhline(0, color="gray")
    ax.axvline(pos, color="gray", linestyle=":")
    ax.set_ylabel("residual / σ")
    ax.legend()
ax_g.set_title(f"reflection {REFLECTION} of pattern {PATTERN}: residuals of the two profiles")
ax_pv.set_xlabel("2θ (deg)")
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** In two or three sentences, say which profile the residuals justify and why, and whether your call depends on the profile. If your pseudo-Voigt call is undecided, say what you can about the cause: residuals that rise at the window's edges, where the cell names a neighbour, point at that line's tail, and a standard error that reaches the certified edge on a weak line points at the line itself, not the method. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the analyst, does your line-broadening method pass validation on your reflection? Cite your L ± 2 se against the certified interval with the margin to its nearer edge, each profile's reduced chi-squared, and what the Gaussian would have certified, and say what the laboratory does next and what the wrong call would cost: every size it reports biased, or a method that works sent back for revalidation. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your data. Your prompt record from section 2 is the natural source; the reason has to cite a value from your output or a line of the certificate table, not a general rule. This session's AI practice, named in the notes, is *specific prompts*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"SciPy      {scipy.__version__}")
print(f"Matplotlib {matplotlib.__version__}")